# Baseline Evaluation – Error Analysis

This notebook analyses the detailed RAG evaluation results produced during
Stage 1 (Baseline Evaluation).  
The goal is to identify weaknesses in retrieval, generation, and grounding,
and develop hypotheses for improving the system.


In [4]:
import pandas as pd

# Load your detailed evaluation results
results_df = pd.read_csv("/Users/cmontefusco/Documents/DSAI2/Rag_project/evaluation/evaluation_results/baseline_evaluation_detailed_2025-12-02_10-16-05.csv")

results_df.head()

,user_input,retrieved_contexts,response,reference,faithfulness,context_precision,context_recall,chunk_size,chunk_overlap
0,Qual é o objetivo principal da RDC 09/2015?,['ADVERTÊNCIA \nEste texto não substitui o pu...,"A Resolução da Diretoria Colegiada (RDC) nº 9,...",O objetivo da RDC 09/2015 é definir os procedi...,1.0,1.0,0.5,800,120
1,Quais etapas de desenvolvimento clínico exigem...,['33. O DDCM pode ser apresentado à Anvisa em...,O Dossiê de Desenvolvimento Clínico de Medicam...,Todos os ensaios clínicos que tenham parte do ...,1.0,0.0,0.5,800,120


## Overview of Metrics

The dataset contains the following evaluation metrics:

- **faithfulness** — how well the answer sticks to retrieved context
- **context_precision** — proportion of retrieved context that is relevant
- **context_recall** — proportion of relevant context successfully retrieved

Below we identify the lowest-scoring examples for each metric.

In [5]:
# Sort by lowest metrics
worst_faithfulness = results_df.sort_values("faithfulness").iloc[0]
worst_precision = results_df.sort_values("context_precision").iloc[0]
worst_recall = results_df.sort_values("context_recall").iloc[0]

worst_faithfulness, worst_precision, worst_recall

(user_input                  Qual é o objetivo principal da RDC 09/2015?
 retrieved_contexts    ['ADVERTÊNCIA  \nEste texto não substitui o pu...
 response              A Resolução da Diretoria Colegiada (RDC) nº 9,...
 reference             O objetivo da RDC 09/2015 é definir os procedi...
 faithfulness                                                        1.0
 context_precision                                                   1.0
 context_recall                                                      0.5
 chunk_size                                                          800
 chunk_overlap                                                       120
 Name: 0, dtype: object,
 user_input            Quais etapas de desenvolvimento clínico exigem...
 retrieved_contexts    ['33. O DDCM pode ser apresentado à Anvisa  em...
 response              O Dossiê de Desenvolvimento Clínico de Medicam...
 reference             Todos os ensaios clínicos que tenham parte do ...
 faithfulness             

# Investigating Low Context Recall (Worst Failure)

This section analyses the example with the *lowest context recall score*.
A low score indicates that the retrieval system failed to surface the
information needed for the model to answer correctly.


In [6]:
row = worst_recall

print("QUESTION:\n", row["user_input"])
print("\nMODEL ANSWER:\n", row["response"])
print("\nGROUND TRUTH:\n", row["reference"])
print("\nRETRIEVED CONTEXTS:\n", row["retrieved_contexts"])
print("\nMETRICS:")
print("faithfulness:", row["faithfulness"])
print("context_precision:", row["context_precision"])
print("context_recall:", row["context_recall"])


QUESTION:
 Qual é o objetivo principal da RDC 09/2015?

MODEL ANSWER:
 A Resolução da Diretoria Colegiada (RDC) nº 9, de 20 de fevereiro de 2015, tem como objetivo definir os procedimentos e requisitos para a realização de ensaios clínicos com medicamentos. Isso inclui a submissão do Dossiê de Desenvolvimento Clínico de Medicamento (DDCM) para aprovação pela Anvisa.

GROUND TRUTH:
 O objetivo da RDC 09/2015 é definir os procedimentos e requisitos para a realização de ensaios clínicos com medicamentos no Brasil, incluindo a submissão do Dossiê de Desenvolvimento Clínico de Medicamento (DDCM). (Fonte: Art. 1º da RDC 09/2015)

RETRIEVED CONTEXTS:
 ['ADVERTÊNCIA  \nEste texto não substitui o publicado no Diário Oficial da União  \n \nMinistério da Saúde  \nAgência Nacional de Vigilância Sanitária  \nRESOLUÇÃO DA DIRETORIA COLEGIA DA-RDC N° 9, DE 20 DE \nFEVEREIRO DE 2015  \nD\ni\ns\np\nõ\ne\n \ns\no\nb\nr\ne\n \no\n \nR\ne\ng\nu\nl\na\nm\ne\nn\nt\no\n \np\na\nr\n\n\na\n \na\n \nr\ne\na\nl\

## 🔎 Investigating the Worst-Case Example (Baseline Evaluation)

To understand the main weakness of our baseline RAG system, we analyse the example with the **lowest context precision** score.  
A precision value of 0.0 indicates that the retrieval stage returned **no relevant context**, meaning the model did not receive the information needed to answer correctly based on the source documents.

---

### **Question**
**“Quais etapas de desenvolvimento clínico exigem submissão de DDCM segundo a RDC 09/2015?”**  
(*Which clinical development stages require submission of the DDCM according to RDC 09/2015?*)

### **Model Answer**
The model responded with a generally correct explanation:

> Phase I, II, and III trials require DDCM submission.  
> Phase IV trials typically do not require DDCM unless used for registration or renewal.

### **Ground Truth**
> All clinical trials conducted in Brazil for registration purposes must submit a DDCM.  
> Phase IV trials do not require DDCM unless they provide data for registration or renewal.

### **Retrieved Context**
The retrieved context contained only a generic paragraph related to the submission of a DDCM, but **did not include the rule explaining which phases require a DDCM**, making it unhelpful for answering the question.

---

## 🧠 Why Did This Example Fail?

- **Context Precision = 0.0**  
  The retrieval step surfaced content that was *not helpful* for answering the question.

- **Context Recall = 0.5**  
  Retrieval found partially related information about DDCM, but not the exact rule needed.

- **Faithfulness = 1.0**  
  The model did not hallucinate; it answered in a consistent and reasonable way despite lacking the correct context.  
  This confirms that the failure lies in **retrieval**, not generation.

- **Root Cause:**  
  The current chunking strategy (`chunk_size = 800`, `chunk_overlap = 120`) likely produces chunks that are too large or too diffuse, causing the retrieval mechanism to select text that is semantically close but not specific enough.

### 📉 Impact on the Chatbot

For a regulatory assistant, incorrect or irrelevant retrieval can:

- Reduce trustworthiness  
- Make it impossible to cite the correct regulation  
- Affect legal or compliance-sensitive use cases  

Improving retrieval is therefore a high priority.

## 🚀 Next Steps to Improve Retrieval and Overall Answer Quality

Based on the worst-case example, the main weakness of the baseline system lies in **low context precision**, meaning the retriever is pulling irrelevant text. Below are the recommended actions to strengthen the RAG pipeline.

---

### **1. Optimise the Chunking Strategy**
Current configuration:
chunk_size = 800
chunk_overlap = 120


This chunk size appears to be **too large**, leading to:

- Mixed topics within the same chunk  
- Difficulty matching specific regulatory rules  
- Lower retrieval precision  

The next step is to compare different chunking strategies in  
`02_chunking_analysis.ipynb` and select the most effective one.

---

### **2. Improve Retrieval with Reranking**

Once chunking is optimised:

- Add a **cross-encoder reranker**  
- Improve ranking of retrieved chunks  
- Reduce noise in the retrieved context

Reranking often significantly increases precision.

---

### **3. Experiment with Query Rewriting (HyDE)**

Query rewriting can help the retriever find the right legal passage, especially when:

- Questions are complex  
- Relevant rules are scattered across the document  
- Terminology varies across articles  

HyDE often boosts recall in regulatory or legal texts.

---

### **4. Update Your RAG Configuration**

After selecting the best-performing chunking strategy, update:

`src/config.py`

```python
CHUNK_SIZE = <your_best_value>
CHUNK_OVERLAP = <your_best_value>
```
This ensures your full chatbot pipeline benefits from the optimisation.

5. Re-run the Baseline Evaluation for Validation

After updating chunking values:

Re-run the baseline evaluation

Compare precision and recall with the previous results

Check if the failing example improves

If the problematic question shows better retrieval, your optimisation is effective.

---

### **🎉 Conclusion**

This analysis shows that the main limitation of the baseline system is retrieval quality, not generation.
By improving chunking, adding reranking, and refining the retrieval process, you can substantially enhance the chatbot’s accuracy and reliability.